# KhaledDSML 23 — Scientific guardrails: Vuong, zero inflation and post-study interpretation
## ضبط فحوص التضخم الصفري ونماذج غير متداخلة وتفسير ما بعد الدراسة

**Development 0.10.0.dev0**, not PyPI stable. Synthetic data only. / جميع البيانات في الأمثلة اصطناعية وتُحسب الإحصاءات فعليًا، مع تحذيرات عربية وإنجليزية.

In [ ]:
%pip install -q "git+https://github.com/noble2013/KhaledDSML.git@feature/v0.10-scientific-final-gap-guards" "scipy>=1.11"

## 1 — van den Broek Poisson zero-excess | فحص زيادة الأصفار

This is NOT the classical Vuong test. The count observations need to be iid and compatible with one Poisson mean. Overdispersion can mimic zero inflation.

In [ ]:
import numpy as np
from scipy import stats
from khaled_dsml import (
    poisson_excess_zero_score, vuong_strictly_nonnested,
    post_study_precision_scenarios, display_analysis_dashboard,
)
rng=np.random.default_rng(2026)
counts=np.where(rng.random(850)<.3,0,rng.poisson(1.8,size=850))
zero=poisson_excess_zero_score(counts)
display_analysis_dashboard(zero,theme="ocean")
print("Upper-tail p (excess zeros):",zero["p_value"])
print("Two-sided chi-square reference:",zero["chi_square_two_sided_p_value"])

## 2 — Vuong ONLY for genuinely strictly nonnested families | Vuong بضوابط صارمة

Explicitly blocks misleading Poisson-vs-ZIP use. Models must be scientifically verified as strictly nonnested and tested on identical independent cases. / لا تستعمله للمقارنة بين Poisson وZIP.

In [ ]:
x=rng.lognormal(.2,.55,350)
ll_lognormal=stats.lognorm.logpdf(x,s=.55,scale=np.exp(.2))
ll_exponential=stats.expon.logpdf(x,scale=x.mean())
result=vuong_strictly_nonnested(
    ll_lognormal,ll_exponential,
    model_a_family="lognormal",model_b_family="exponential",
    relationship="strictly_nonnested",
)
display_analysis_dashboard(result,theme="amethyst")

## 3 — Post-study precision and external-effect scenarios | دقة التقدير وبدائل القوة اللاحقة

Do NOT calculate observed power from the study p-value. Sensitivity is conditional on independently proposed true effects and reuses estimated study SE. / لا نحسب القدرة المرصودة لأنها لا تضيف دليلًا مستقلًا.

In [ ]:
precision=post_study_precision_scenarios(
    estimate=.25,standard_error=.12,
    externally_specified_effects=[0.,.1,.2,.35,.5])
display_analysis_dashboard(precision,theme="aurora")
assert precision["observed_power_computed"] is False

## Research integrity | الضوابط العلمية

- van den Broek's 1995 test describes excess zeros against an **intercept-only iid Poisson**, not definitive proof of a ZIP regression model.
- Wilson (2015) explains why Poisson vs ZIP violates the classical strictly-nonnested Vuong reference.
- Hoenig & Heisey (2001) explain why observed post-hoc power is a redundant function of the completed test result.
- Catalogue 256/256 indicates **candidate-source links, not 256 scientifically complete methods**. Row #85 is a substitute; #117 is deliberately restricted.

References in `docs/ZERO_INFLATION_AND_PRECISION_0_10_AR_EN.md`.